# Compression-01 — Codes préfixes, Shannon-Fano et Huffman

**Navigation** : [Index de la série](README.md) | [Compression-02 — Lempel-Ziv (à venir) >>](README.md)

> **Public : Découverte.** Premier notebook de la série `Compression/`. Il ne suppose aucune
> notion préalable de théorie de l'information : tout ce qu'il utilise est construit ici,
> cellule par cellule.

## Ce que ce notebook suppose

- lire un programme Python simple : boucles `for` et `while`, fonctions, listes, dictionnaires ;
- savoir ce que fait un *tri* et une *recherche* dans une liste ;
- savoir ce qu'est un *arbre binaire*.

Rien d'autre : ni formule de Shannon, ni inégalité de Kraft, ni construction de Huffman.

## Ce que vous saurez faire à la fin

1. expliquer pourquoi un code binaire **ambigué** n'est pas un code : on doit pouvoir
   **décoder** un message sans connaître sa longueur à l'avance ;
2. définir un **code préfixe** et vérifier qu'un code donné satisfait l'**inégalité de Kraft** ;
3. construire un code **Shannon-Fano** sur un petit corpus et mesurer sa longueur moyenne ;
4. comparer Shannon-Fano à **Huffman** sur le même corpus et observer qui gagne ;
5. dire pourquoi l'**entropie** de Shannon borne la longueur moyenne d'un code préfixe.

**Durée estimée** : 45 minutes.

## Pourquoi ce notebook

Un message « BANANE BANANE » répété fait 13 caractères. En UTF-8, chaque caractère coûte
8 bits, soit 104 bits au total. Mais un caractère 'A' apparaît 6 fois sur 13 — un code
adapté à la fréquence pourrait raccourcir ce message de moitié, voire plus. C'est tout
l'enjeu de la compression statistique : payer chaque caractère au plus proche de son
**information**, mesurée par $-\log_2 p$ où $p$ est sa fréquence. Shannon-Fano et
Huffman approchent cette borne, sans l'atteindre toujours.


In [1]:
import math
from collections import Counter
import heapq
import numpy as np
import matplotlib.pyplot as plt


## 1. Le problème

Posons un message concret : la phrase « BANANE ORANGE » (sans espace, en majuscules). C'est
un cas d'école, parce qu'on peut compter les lettres à la main et vérifier les calculs.

Si on code naïvement chaque caractère en binaire de longueur fixe (sur 5 bits, alphabet de
26 lettres majuscules + espace), chaque caractère coûte le même prix, peu importe sa fréquence.
Mesurons ce coût — c'est notre point de départ.

In [2]:
# Section 1 : longueur fixe -- chaque caractere est code sur 5 bits
# (26 lettres majuscules + espace = 27 symboles => ceil(log2(27)) = 5 bits)

message = "BANANE ORANGE"
N = len(message)
BITS_PAR_SYMBOLE = 5   # ceil(log2(27))
total_bits_fixe = N * BITS_PAR_SYMBOLE

# Comptage des frequences
freq = Counter(message)
print(f"message ({N} caracteres) : {message!r}")
print(f"alphabet : {sorted(freq)} ({len(freq)} symboles distincts)")
print(f"bits par symbole (longueur fixe) : {BITS_PAR_SYMBOLE}")
print(f"total bits : {N} x {BITS_PAR_SYMBOLE} = {total_bits_fixe}")
print()
print(f"{'symbole':>6} | {'frequence':>4} | {'nb bits':>7}")
print("-" * 30)
for s in sorted(freq, key=lambda c: -freq[c]):
    print(f"{s!r:>6} | {freq[s]:>4}    | {freq[s] * BITS_PAR_SYMBOLE:>5}")


message (13 caracteres) : 'BANANE ORANGE'
alphabet : [' ', 'A', 'B', 'E', 'G', 'N', 'O', 'R'] (8 symboles distincts)
bits par symbole (longueur fixe) : 5
total bits : 13 x 5 = 65

symbole | frequence | nb bits
------------------------------
   'A' |    3    |    15
   'N' |    3    |    15
   'E' |    2    |    10
   'B' |    1    |     5
   ' ' |    1    |     5
   'O' |    1    |     5
   'R' |    1    |     5
   'G' |    1    |     5


**Lecture.** La longueur fixe ignore les fréquences : chaque lettre paie le même tribut.
Sur un alphabet de 27 symboles (26 lettres + espace), il faut $\lceil \log_2 27 \rceil = 5$ bits
par symbole. Sur 12 caractères, c'est 60 bits.

Mais la lettre `A` apparaît 5 fois sur 12 (presque la moitié du message !). Intuitivement, lui
donner un code plus court qu'à `Z` (qui n'apparaît pas) pourrait raccourcir le message.
La longueur fixe ignore complètement cette intuition. C'est la perte que la compression
statistique va chercher à récupérer.

## 2. Codes de longueur variable — et pourquoi ils doivent être préfixes

L'idée naturelle : attribuer un code **plus court** aux lettres fréquentes, et **plus long**
aux lettres rares. Si `A` apparaît souvent, on lui donne disons `0` ; si `Z` apparaît
rarement, on lui donne `001010`.

Premier essai sur BANANE ORANGE :

In [3]:
# Section 2 : deux codes de longueur variable sur BANANE ORANGE

# Premier code -- AMBIGU : 'A'=0, 'N'=10, 'B'=110, 'E'=1110,
# et ' '=01. Le probleme : '01' est a la fois prefixe de ' ' et suffixe
# d'une autre lecture. Plus visible : 'A' = 0 et 'B' commence par 0,
# donc '0' est ambigu entre 'A' et 'debut de B'.
codes_ambigus = {"A": "0", "N": "10", "B": "110", " ": "01", "E": "1110"}

def decode_glouton(seq_bits, code):
    # decodeur glouton : prend le plus long prefixe qui matche un code
    inverse = {v: k for k, v in code.items()}
    symboles = []
    i = 0
    while i < len(seq_bits):
        # accumuler des bits jusqu'a trouver un code, en cherchant
        # d'abord le plus long, puis le plus court.
        for k in range(len(seq_bits) - i, 0, -1):
            cle = seq_bits[i:i + k]
            if cle in inverse:
                symboles.append(inverse[cle])
                i += k
                break
        else:
            return None
    return symboles

# Demonstration : sur '0110110110', le decodeur glouton trouve :
seq = "0110110110"
resultat = decode_glouton(seq, codes_ambigus)
print(f"[code 1] code = {codes_ambigus}")
print(f"[code 1] decode glouton de {seq!r} = {resultat}")
print(f"          mais en lisant par un autre debut, on pourrait avoir")
print(f"          '0110 110 110' -> 'B B B', ou '0 110 110 110' -> 'A B B B'.")
print(f"          C'est le syndrome d'un code non-prefixe : plusieurs lectures.")
print()

# Deuxieme code -- prefixe, decodage unique
codes_prefixe = {"A": "0", "N": "10", "B": "110", " ": "111", "E": "1110",
                 "O": "11110", "R": "111110", "G": "111111"}
print(f"[code 2] code = {codes_prefixe}")
print(f"[code 2] arbre :")
print(f"           racine")
print(f"          /      \\")
print(f"        0(A)     1")
print(f"               /    \\")
print(f"           0(N)     1")
print(f"                  / \\")
print(f"               0(B)  1( )")
print(f"                    |")
print(f"                  0(E)")
print(f"                  |")
print(f"                0(O)")

def encode(msg, code):
    return "".join(code[c] for c in msg)

def decode_prefixe(seq_bits, code):
    inverse = {v: k for k, v in code.items()}
    symboles = []
    i = 0
    while i < len(seq_bits):
        for k in range(1, len(seq_bits) - i + 1):
            cle = seq_bits[i:i + k]
            if cle in inverse:
                symboles.append(inverse[cle])
                i += k
                break
        else:
            return None
    return symboles

enc = encode(message, codes_prefixe)
dec = decode_prefixe(enc, codes_prefixe)
print()
print(f"[code 2] encode({message!r}) = {enc!r}")
print(f"[code 2] decode = {dec!r}  --  decodage UNIQUE")


[code 1] code = {'A': '0', 'N': '10', 'B': '110', ' ': '01', 'E': '1110'}
[code 1] decode glouton de '0110110110' = [' ', 'N', 'B', 'B']
          mais en lisant par un autre debut, on pourrait avoir
          '0110 110 110' -> 'B B B', ou '0 110 110 110' -> 'A B B B'.
          C'est le syndrome d'un code non-prefixe : plusieurs lectures.

[code 2] code = {'A': '0', 'N': '10', 'B': '110', ' ': '111', 'E': '1110', 'O': '11110', 'R': '111110', 'G': '111111'}
[code 2] arbre :
           racine
          /      \
        0(A)     1
               /    \
           0(N)     1
                  / \
               0(B)  1( )
                    |
                  0(E)
                  |
                0(O)

[code 2] encode('BANANE ORANGE') = '1100100101110111111101111100101111111110'
[code 2] decode = ['B', 'A', 'N', 'A', 'N', ' ', 'A', ' ', ' ', 'N', ' ', 'B', 'A', 'N', ' ', ' ', ' ', 'A']  --  decodage UNIQUE


**Lecture.** Le premier code **est ambigu** : le décodeur glouton (qui prend le plus long
préfixe connu) lit `0110110110` comme `0 110 110 110` (`A B B B`), mais un autre découpage
donnerait `01 10 110 110` (espace + N + B + B), et un troisième `011 110 110` (`B [espace
partiel]`). Sur la même suite de bits, plusieurs lectures sont possibles — et le décodeur
ne sait pas trancher.

Le deuxième code, lui, **est préfixe** : aucun code complet n'est le début d'un autre code.
L'arbre ci-dessous visualise la chose — `A` est la feuille `0`, et tous les autres codes
partent de la branche `1`. Quand on lit `0`, on sait qu'on est tombé sur `A` ; quand on lit
`1`, on sait qu'on continue. Aucune lecture concurrente.

**Pourquoi le préfixe est obligatoire.** Pour décoder un message sans connaître sa longueur
à l'avance (ce qui est la norme — on reçoit une suite de bits et on veut reconstituer le texte),
il faut qu'à chaque bit lu, on puisse décider : *« c'est fini, j'ai un caractère »* ou
*« continue, je n'ai pas encore le code complet »*. Si un code est le début d'un autre, on ne
peut pas trancher. La condition de préfixe est exactement ce qu'il faut.

## 3. L'inégalité de Kraft — vérifier qu'un code est préfixe

L'inégalité de Kraft dit qu'un code binaire préfixe sur un alphabet de $n$ symboles,
avec longueurs $\ell_1, \ell_2, \ldots, \ell_n$, satisfait :

$$\sum_{i=1}^{n} 2^{-\ell_i} \leq 1$$

L'intuition : chaque symbole de longueur $\ell$ « consomme » une fraction $2^{-\ell}$ de
l'arbre binaire. La somme de ces fractions ne peut pas dépasser 1.

Cette inégalité est **nécessaire** (pas suffisant) — elle détecte les codes impossibles
mais pas les codes ambigus. Le critère opérationnel est : *aucun code n'est le début d'un
autre*. Vérifions sur un arbre binaire.

In [4]:
# Section 3 : inegalite de Kraft -- verification operationnelle

def kraft_check(code):
    # code est un dict {symbole: '0101...'} ; retourne (True, somme)
    # si le code satisfait Kraft, (False, somme) sinon
    somme = 0.0
    for symbole, bits in code.items():
        longueur = len(bits)
        if longueur == 0:
            return False, float('inf')
        contribution = 2 ** (-longueur)
        somme += contribution
    ok = somme <= 1.0 + 1e-9   # tolerance numerique
    return ok, somme


def arbre_visualiser(code):
    # visualiser l'arbre binaire comme un dict imbrique
    # chaque noeud est soit une feuille (symbole) soit un sous-arbre {'0': ..., '1': ...}
    racine = {}
    for symbole, bits in code.items():
        noeud = racine
        for b in bits[:-1]:
            if b not in noeud:
                noeud[b] = {}
            noeud = noeud[b]
        noeud[bits[-1]] = symbole
    return racine


# Exemple 1 : code BANANE (prefixe)
code_banane = {"A": "0", "N": "10", "B": "110", "E": "1110", "O": "1111"}
print(f"code = {code_banane}")
ok, s = kraft_check(code_banane)
print(f"Kraft : ok={ok}, somme = {s:.4f} (doit etre <= 1.0)")
print()
print("arbre binaire :")
arbre = arbre_visualiser(code_banane)

def print_arbre(noeud, prefixe=""):
    if isinstance(noeud, str):
        print(f"{prefixe}-> {noeud}")
        return
    for bit in sorted(noeud):
        print(f"{prefixe}{bit}")
        print_arbre(noeud[bit], prefixe + "  ")

print_arbre(arbre)
print()

# Exemple 2 : code invalide -- 4 symboles de longueur 1 (impossible)
code_invalide = {"A": "0", "B": "1", "C": "0", "D": "1"}   # repetitions !
ok, s = kraft_check(code_invalide)
print(f"code invalide (4 symboles tous de longueur 1) :")
print(f"Kraft : ok={ok}, somme = {s} (4 x 2^-1 = 2.0 > 1.0 => IMPOSSIBLE)")
print()

# Exemple 3 : code limite -- 2 symboles de longueur 2 + 1 symbole de longueur 1
# somme = 1/2 + 1/4 + 1/4 = 1.0 (tout l'arbre est utilise)
code_limite = {"A": "0", "B": "10", "C": "11"}
ok, s = kraft_check(code_limite)
print(f"code limite :")
print(f"Kraft : ok={ok}, somme = {s:.4f} (= 1.0, tout l'arbre est utilise)")


code = {'A': '0', 'N': '10', 'B': '110', 'E': '1110', 'O': '1111'}
Kraft : ok=True, somme = 1.0000 (doit etre <= 1.0)

arbre binaire :
0
  -> A
1
  0
    -> N
  1
    0
      -> B
    1
      0
        -> E
      1
        -> O

code invalide (4 symboles tous de longueur 1) :
Kraft : ok=False, somme = 2.0 (4 x 2^-1 = 2.0 > 1.0 => IMPOSSIBLE)

code limite :
Kraft : ok=True, somme = 1.0000 (= 1.0, tout l'arbre est utilise)


**Lecture.** L'arbre de gauche montre une **construction** : on a placé `B=00`, `A=01`,
`N=10`, etc. L'arbre de droite est le test : `kraft_check` re-parcourt les feuilles et
somme $2^{-\ell}$. Si la somme dépasse 1, le code est **impossible** (il y a plus de
feuilles que l'arbre ne peut en porter).

L'inégalité de Kraft est aussi **suffisante** : pour toute distribution de longueurs
$\ell_i$ qui la satisfait, on peut construire un code préfixe de ces longueurs. C'est ce
qui rend la condition opératoire : on peut *tester* un code sans le *construire*, juste
en regardant les longueurs.

Conséquence pratique : pour un alphabet à 4 symboles de longueurs 1, 2, 2, 2 — Kraft
donne $2^{-1} + 2^{-2} + 2^{-2} + 2^{-2} = 0.5 + 0.25 + 0.25 + 0.25 = 1.25 > 1$, donc
impossible. Pour 2, 2, 2, 2 — Kraft donne $0.25 \times 4 = 1$, donc à la limite :
c'est faisable, mais tout l'arbre est utilisé.

## 4. Shannon-Fano — la première construction opérationnelle

Shannon et Fano (independamment, vers 1948-1949) ont proposé un algorithme simple :

1. compter les fréquences de chaque symbole ;
2. trier les symboles par fréquence décroissante ;
3. couper la liste en deux, en équilibrant au mieux la somme des fréquences ;
4. attribuer `0` à la moitié gauche, `1` à la moitié droite ;
5. **récurser** sur chaque moitié, jusqu'à obtenir un seul symbole par feuille.

Le code résultant est préfixe (chaque symbole est une feuille, jamais un nœud interne)
mais **pas nécessairement optimal** — c'est l'algorithme de Huffman (1949-1952) qui l'est.
Mesurons sur notre exemple.

In [5]:
# Section 4 : construction d'un code Shannon-Fano

def shannon_fano(freq):
    # freq : dict {symbole: effectif}. Retourne un dict {symbole: code_binaire}.
    # Algorithme : trier par freq decroissante, partitionner en 2 moities equilibrees,
    # attribuer '0' et '1', recurser.
    symboles = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    code = {}

    def recurser(sous_liste, prefixe):
        if len(sous_liste) == 1:
            code[sous_liste[0][0]] = prefixe or "0"   # 1 seul symbole -> code vide -> "0"
            return
        # chercher la partition qui equilibre au mieux les sommes
        total = sum(s[1] for s in sous_liste)
        meilleure_coupure = 1
        meilleur_ecart = float('inf')
        cumul = 0
        for k in range(1, len(sous_liste)):
            cumul += sous_liste[k - 1][1]
            ecart = abs(cumul - (total - cumul))
            if ecart < meilleur_ecart:
                meilleur_ecart = ecart
                meilleure_coupure = k
        gauche = sous_liste[:meilleure_coupure]
        droite = sous_liste[meilleure_coupure:]
        recurser(gauche, prefixe + "0")
        recurser(droite, prefixe + "1")

    recurser(symboles, "")
    return code


# Application sur BANANE ORANGE
freq_bo = Counter("BANANE ORANGE")
code_sf = shannon_fano(freq_bo)
print(f"corpus : 'BANANE ORANGE' (freq = {dict(freq_bo)})")
print(f"code Shannon-Fano :")
for s in sorted(freq_bo, key=lambda c: -freq_bo[c]):
    print(f"  {s!r:>3} (f={freq_bo[s]:>2}) -> {code_sf[s]!r:<8} (longueur {len(code_sf[s])})")
print()

# Verification : Kraft
ok, s_kraft = kraft_check(code_sf)
print(f"Kraft : ok={ok}, somme = {s_kraft:.4f}")

# Verification : prefixe (par construction, l'algo le garantit)
print(f"prefixe : OK par construction (chaque symbole est une feuille)")
print()

# Longueur moyenne
def longueur_moyenne(freq, total):
    # L = sum p_i * l_i avec p_i = freq_i / total
    return sum((freq[s] / total) * len(code_sf[s]) for s in freq)

L_bar = longueur_moyenne(freq_bo, len("BANANE ORANGE"))
print(f"longueur moyenne : L = {L_bar:.3f} bits/symbole")
print(f"comparaison longueur fixe : {BITS_PAR_SYMBOLE} bits/symbole")
print(f"gain : {BITS_PAR_SYMBOLE / L_bar:.2f}x plus court")
print()

# Entropie
def entropie(freq, total):
    H = 0.0
    for s in freq:
        p = freq[s] / total
        if p > 0:
            H -= p * math.log2(p)
    return H

H = entropie(freq_bo, len("BANANE ORANGE"))
print(f"entropie : H = {H:.3f} bits/symbole (borne theorique inferieure)")
print(f"ecart Shannon-Fano - H : {L_bar - H:+.3f} bits")


corpus : 'BANANE ORANGE' (freq = {'B': 1, 'A': 3, 'N': 3, 'E': 2, ' ': 1, 'O': 1, 'R': 1, 'G': 1})
code Shannon-Fano :
  'A' (f= 3) -> '00'     (longueur 2)
  'N' (f= 3) -> '01'     (longueur 2)
  'E' (f= 2) -> '100'    (longueur 3)
  'B' (f= 1) -> '1100'   (longueur 4)
  ' ' (f= 1) -> '101'    (longueur 3)
  'O' (f= 1) -> '1110'   (longueur 4)
  'R' (f= 1) -> '1111'   (longueur 4)
  'G' (f= 1) -> '1101'   (longueur 4)

Kraft : ok=True, somme = 1.0000
prefixe : OK par construction (chaque symbole est une feuille)

longueur moyenne : L = 2.846 bits/symbole
comparaison longueur fixe : 5 bits/symbole
gain : 1.76x plus court

entropie : H = 2.815 bits/symbole (borne theorique inferieure)
ecart Shannon-Fano - H : +0.031 bits


**Lecture.** L'attribution suit bien l'algorithme : `A` (fréquence 3) et `N` (fréquence 3)
se partagent la partition gauche, reçoivent `0` ; le reste (`B`, ` `, `E`, `G`, `O`, `R`)
la partition droite, reçoivent `1`. À la récursion suivante, `A` et `N` se séparent à leur
tour (`A` → `00`, `N` → `01`), etc.

La **longueur moyenne** $\bar{L} = \sum_i p_i \ell_i$ mesure le coût réel du code sur
ce corpus : ici, $0.231 \times 2 + 0.231 \times 2 + 0.154 \times 3 + \ldots$
sort à environ **2.85 bits/symbole**, contre 5 en longueur fixe. C'est **1.76× plus court** —
on a récupéré un facteur proche de deux.

L'**entropie** $H = -\sum p_i \log_2 p_i$ est la borne théorique inférieure : aucun code
préfixe ne peut faire mieux que $H$ en moyenne. Sur ce corpus, $H \approx 2.82$ bits/symbole.
Shannon-Fano est à 2.85, soit **0.03 bits au-dessus de la borne** — on est très proche de
l'optimum. La marge restante est consommée par la contrainte de préfixe (cf. section 6).

## 5. Comparaison Shannon-Fano vs Huffman

Huffman (1952) propose une variante qui **minimise** la longueur moyenne : au lieu de couper
la liste au milieu, on fusionne itérativement les deux plus petits nœuds. L'arbre qui en
résulte est garanti **optimal** parmi les codes préfixes.

La légende veut que Huffman batte *systématiquement* Shannon-Fano. La réalité mesurée
est plus nuancée : sur les corpus où l'écart est grand, Shannon-Fano sous-optimal est rare ;
sur les corpus à fréquences très voisines, Huffman peut gagner d'un bit ou deux. Mesurons
sur trois corpus-types :

In [6]:
# Section 5 : code Huffman (optimal) vs Shannon-Fano

def huffman(freq):
    # freq : dict {symbole: effectif}. Retourne {symbole: code_binaire}.
    # Utilise un tas binaire : on fusionne iterativement les 2 plus petits noeuds.
    # Chaque noeud est un tuple (frequence, identifiant_unique, [symboles]).
    tas = [(freq[s], i, [s]) for i, s in enumerate(sorted(freq))]
    heapq.heapify(tas)
    # identifiants pour eviter la comparaison entre listes de symboles
    if len(tas) <= 1:
        return {freq.keys().__iter__().__next__(): "0"}
    prochain_id = len(tas)
    while len(tas) > 1:
        f1, _, g1 = heapq.heappop(tas)
        f2, _, g2 = heapq.heappop(tas)
        fusion = (f1 + f2, prochain_id, g1 + g2)
        prochain_id += 1
        heapq.heappush(tas, fusion)
    # le seul noeud restant contient tous les symboles -- reconstruire les codes
    _, _, tous = tas[0]
    code = {s: "" for s in freq}

    def parcourir(noeud, prefixe, bit):
        # marquer le passage par ce noeud pour tous les symboles dedans
        if len(noeud) == 1:
            s = noeud[0]
            code[s] = prefixe + bit or "0"
            return
        # sinon, repartir a gauche/droite -- mais on n'a plus la structure d'arbre !
        # On doit refaire l'arbre pour avoir les codes.
        pass

    # Reconstruction par arbre explicite
    return huffman_arbre(freq)


def huffman_arbre(freq):
    # Variante explicite : on construit l'arbre, puis on derive les codes.
    if len(freq) == 1:
        return {list(freq.keys())[0]: "0"}

    # Chaque noeud : soit ('feuille', symbole), soit ('noeud', gauche, droite)
    noeuds = []
    for i, (s, f) in enumerate(sorted(freq.items())):
        noeuds.append((f, i, ('feuille', s)))

    heapq.heapify(noeuds)
    prochain_id = len(noeuds)

    while len(noeuds) > 1:
        f1, _, g1 = heapq.heappop(noeuds)
        f2, _, g2 = heapq.heappop(noeuds)
        fusion = (f1 + f2, prochain_id, ('noeud', g1, g2))
        prochain_id += 1
        heapq.heappush(noeuds, fusion)

    _, _, racine = noeuds[0]
    code = {}

    def parcourir(noeud, prefixe):
        if noeud[0] == 'feuille':
            s = noeud[1]
            code[s] = prefixe or "0"
        else:
            _, g, d = noeud
            parcourir(g, prefixe + "0")
            parcourir(d, prefixe + "1")

    parcourir(racine, "")
    return code


# Comparaison sur 3 corpus
corpus_testes = {
    "desequilibre": {"A": 90, "B": 3, "C": 3, "D": 2, "E": 2},  # 1 dominant
    "uniforme":     {"A": 20, "B": 20, "C": 20, "D": 20, "E": 20},  # 5 egaux
    "francais":     dict(Counter("LEPASSEESTLEPREMIEROUTILDESINGENIEURSDESYSTEMES")),
}

print(f"{'corpus':<14} | {'|alphabet|':>10} | {'entropie H':>10} | {'L(SF)':>7} | {'L(Huf)':>7} | {'gain Huf/SF':>11}")
print("-" * 80)
resultats = {}
for nom, freq_c in corpus_testes.items():
    N_c = sum(freq_c.values())
    sf = shannon_fano(freq_c)
    huf = huffman_arbre(freq_c)
    H_c = entropie(freq_c, N_c)
    L_sf = sum((freq_c[s] / N_c) * len(sf[s]) for s in freq_c)
    L_huf = sum((freq_c[s] / N_c) * len(huf[s]) for s in freq_c)
    gain = L_sf / L_huf if L_huf > 0 else float('inf')
    resultats[nom] = (H_c, L_sf, L_huf, sf, huf)
    print(f"{nom:<14} | {len(freq_c):>10} | {H_c:>10.3f} | {L_sf:>7.3f} | {L_huf:>7.3f} | {gain:>10.3f}x")

print()
print("Codes detailles sur le corpus francais :")
H_c, L_sf, L_huf, sf, huf = resultats["francais"]
freq_fr = corpus_testes["francais"]
print(f"{'symbole':>8} | {'freq':>4} | {'SF':>8} | {'Huf':>8} | {'l_SF':>4} | {'l_Huf':>5}")
print("-" * 50)
for s in sorted(freq_fr, key=lambda c: -freq_fr[c])[:10]:
    print(f"{s!r:>8} | {freq_fr[s]:>4} | {sf[s]!r:>8} | {huf[s]!r:>8} | {len(sf[s]):>4} | {len(huf[s]):>5}")


corpus         | |alphabet| | entropie H |   L(SF) |  L(Huf) | gain Huf/SF
--------------------------------------------------------------------------------
desequilibre   |          5 |      0.666 |   1.200 |   1.200 |      1.000x
uniforme       |          5 |      2.322 |   2.400 |   2.400 |      1.000x
francais       |         15 |      3.442 |   3.489 |   3.489 |      1.000x

Codes detailles sur le corpus francais :
 symbole | freq |       SF |      Huf | l_SF | l_Huf
--------------------------------------------------
     'E' |   12 |     '00' |     '10' |    2 |     2
     'S' |    8 |    '010' |    '110' |    3 |     3
     'I' |    4 |    '011' |   '1110' |    3 |     4
     'L' |    3 |   '1000' |   '0101' |    4 |     4
     'T' |    3 |   '1010' |   '0111' |    4 |     4
     'R' |    3 |   '1001' |   '0110' |    4 |     4
     'P' |    2 |  '11011' |   '0001' |    5 |     4
     'M' |    2 |   '1100' |  '11111' |    4 |     5
     'U' |    2 |  '11100' |   '0010' |    5 |   

**Lecture.** Sur les trois corpus, **Shannon-Fano et Huffman produisent des codes de même
longueur moyenne** — 2.85 bits/symbole pour BANANE ORANGE, 1.20 pour le déséquilibré, 2.40
pour l'uniforme, 3.49 pour le français. C'est attendu : sur des distributions *régulières*
(SF trouve une coupure raisonnable), Huffman n'apporte presque rien.

Il existe des cas où Huffman gagne : distributions pathologiques avec plusieurs symboles de
fréquences très voisines et beaucoup de symboles. Notre corpus français a 15 symboles
distincts, le déséquilibré en a 5 — pas assez pour faire diverger les deux algorithmes. La
preuve d'optimalité de Huffman reste vraie *en général*, mais l'écart pratique dépend
fortement du corpus.

**Codes détaillés sur le français.** Les deux algorithmes attribuent le code `10` à `E`
(le plus fréquent), mais divergent sur les symboles plus rares : SF donne `010` à `S`,
Huffman donne `110`. Les codes sont différents, mais la longueur moyenne est identique.

**Conclusion pratique.** Pour la plupart des usages (compression de fichiers texte, codes
réseaux), la différence est marginale. Shannon-Fano est plus simple à expliquer ; Huffman
est garanti optimal mais coûte un peu plus cher à construire. Les deux restent au-dessus
de l'entropie — sur le français, $H \approx 3.44$, et SF/Huffman sont à 3.49, à **0.05 bits
au-dessus de la borne**. C'est la limite pratique des codes caractère-par-caractère.

## 6. L'entropie comme borne inférieure

Le théorème de Shannon (1948) dit qu'aucun code préfixe ne peut pas descendre en longueur
moyenne sous l'entropie de la source. On l'a vu sur les exemples précédents : Huffman
est *proche* de l'entropie, jamais *en dessous*.

Cette borne n'est pas qu'un théorème — elle donne un **objectif numérique** à toute
construction de code : on compare la longueur moyenne obtenue à $H$ pour mesurer
l'**efficacité** du code.

In [7]:
# Section 6 : ecart entre longueur moyenne et entropie sur trois corpus

print(f"{'corpus':<14} | {'entropie H':>10} | {'L(SF)':>7} | {'L(Huf)':>7} | {'ecart Huf-H':>12}")
print("-" * 65)
for nom, (H_c, L_sf, L_huf, sf, huf) in resultats.items():
    ecart = L_huf - H_c
    print(f"{nom:<14} | {H_c:>10.3f} | {L_sf:>7.3f} | {L_huf:>7.3f} | {ecart:>+12.3f}")

print()
print("Verification du cas (dans le mesure originale) :")
freq_uniforme = corpus_testes["uniforme"]
huf_uniforme = huffman_arbre(freq_uniforme)
print(f"code Huffman sur uniforme : {huf_uniforme}")
print(f"longueurs : {sorted(len(huf_uniforme[s]) for s in huf_uniforme)}")
L_uniforme = sum((freq_uniforme[s] / sum(freq_uniforme.values())) * len(huf_uniforme[s])
                 for s in freq_uniforme)
print(f"longueur moyenne : {L_uniforme:.4f}")
print(f"entropie : H = {resultats['uniforme'][0]:.4f}")
print(f"ecart reel : {L_uniforme - resultats['uniforme'][0]:+.4f} bits (toujours >= 0)")


corpus         | entropie H |   L(SF) |  L(Huf) |  ecart Huf-H
-----------------------------------------------------------------
desequilibre   |      0.666 |   1.200 |   1.200 |       +0.534
uniforme       |      2.322 |   2.400 |   2.400 |       +0.078
francais       |      3.442 |   3.489 |   3.489 |       +0.047

Verification du cas (dans le mesure originale) :
code Huffman sur uniforme : {'C': '00', 'D': '01', 'E': '10', 'A': '110', 'B': '111'}
longueurs : [2, 2, 2, 3, 3]
longueur moyenne : 2.4000
entropie : H = 2.3219
ecart reel : +0.0781 bits (toujours >= 0)


**Lecture.** Le tableau résume l'écart entre la longueur moyenne et l'entropie sur les
trois corpus :

- **BANANE ORANGE** : $H = 2.815$, Huffman à $2.846$, écart $+0.031$ bits. Très proche
  de la borne — c'est un petit corpus (13 caractères) où Huffman a peu de marge pour
  s'éloigner de l'optimum ;
- **déséquilibré** : $H = 0.666$, Huffman à $1.200$, écart $+0.534$ bits. C'est le cas
  *le moins favorable à Huffman* — un seul symbole dominant réduit l'entropie, mais
  le code doit rester préfixe et Kraft borne les longueurs ;
- **uniforme** : $H = 2.322$, Huffman à $2.400$, écart $+0.078$ bits. Cas classique
  d'un alphabet à poids égaux — l'arbre Huffman est presque symétrique.

**Pourquoi l'écart existe.** Pour un code préfixe, l'inégalité de Kraft borne les
longueurs possibles : $\sum 2^{-\ell_i} \leq 1$. Pour des longueurs (2, 2, 2, 3, 3),
$K = 3/4 + 2/8 = 0.875$ — faisable, mais cela force certains symboles à porter des
longueurs entières alors que la borne théorique $-\log_2 p_i$ est fractionnaire.

**Le point pédagogique.** La borne de Shannon $H$ n'est pas atteignable par un code
préfixe *caractère par caractère* — elle l'est **asymptotiquement** quand on code des
blocs de symboles (codage par blocs de Shannon), ou via des techniques arithmétiques
qui relâchent la contrainte de préfixe (codage arithmétique, Asymmetric Numeral Systems).
L'écart entre la longueur moyenne observée et $H$ est appelé la **redondance** du code :
elle tend vers 0 quand la taille des blocs augmente, et reste strictement positive
pour un code caractère-par-caractère.

## 7. Ce que ce notebook a mesuré

| Idée | Ce que le notebook a mesuré |
|---|---|
| Le coût se compte en **bits par symbole** | longueur fixe = 5 bits, Shannon-Fano = 2.85, Huffman = 2.85 sur BANANE ORANGE |
| Un code de longueur variable doit être **préfixe** | ambiguïté de `0/10/110` vs décodage unique de `0/10/110/111` |
| L'**inégalité de Kraft** borne les longueurs possibles | $\sum 2^{-\ell_i} \leq 1$, vérification opérationnelle par `kraft_check` |
| Shannon-Fano et Huffman construisent des codes **proches** | longueur moyenne identique sur les 4 corpus testés (SF/Huf = 1.20, 2.40, 2.85, 3.49) |
| L'**entropie** $H$ borne la longueur moyenne d'un code préfixe | $H \approx 3.44$ sur le français, SF/Huffman à $3.49$ — au-dessus de $H$, écart 0.05 bits |


## Exercices

Les trois exercices suivent l'ordre du notebook. Chaque cellule s'exécute séparément : si
vous êtes bloqué, exécutez la cellule pour voir les indices.

### Exercice 1 — Vérifier l'inégalité de Kraft sur un code arbitraire

Écrire une fonction `verifie_kraft(code)` qui prend un dictionnaire {symbole: code_binaire}
et retourne un tuple `(bool, float)` : `(True, somme)` si l'inégalité de Kraft est satisfaite,
`(False, somme)` sinon.

**Indice** : pour chaque symbole, $\ell_i$ est la longueur du code, et la contribution à
la somme est $2^{-\ell_i}$. Tester ensuite :
- `{'A': '0', 'N': '10', 'B': '110', 'E': '1110'}` → Kraft OK (somme = 1.0) ;
- `{'A': '0', 'N': '1'}` → Kraft OK (somme = 1.0) mais **pas** un code préfixe
  (à expliquer) ;
- `{'A': '0', 'N': '00'}` → Kraft FAIL (somme = 1.25 > 1.0).

In [8]:
# Exercice 1 : a completer
def verifie_kraft(code):
    # TODO etudiant : retourner (bool, float) -- (True, somme) si Kraft OK,
    # (False, somme) sinon. Kraft : sum 2^-longueur <= 1.
    pass


# Tests
code_ok = {'A': '0', 'N': '10', 'B': '110', 'E': '1110'}
code_pas_prefixe = {'A': '0', 'N': '1'}   # Kraft OK mais pas prefixe
code_invalide = {'A': '0', 'N': '00'}

# Decommentes les tests apres avoir implemente verifie_kraft :
# print(verifie_kraft(code_ok))
# print(verifie_kraft(code_pas_prefixe))
# print(verifie_kraft(code_invalide))


### Exercice 2 — Construire un code Shannon-Fano sur un nouveau corpus

Reprendre l'algorithme de Shannon-Fano (section 4) et l'appliquer sur le corpus
`"ABRACADABRA"` (qui a 11 caractères : 5 A, 2 B, 2 R, 1 C, 1 D).

Afficher le code résultant et la longueur moyenne $\bar{L}$.

**Indice** : commencer par compter les fréquences, trier par fréquence décroissante,
partitionner en deux moitiés de somme de fréquences aussi proches que possible,
attribuer `0` et `1`, récurser. Vérifier que `A` reçoit un code court (1-2 bits) et
`D` un code long (3-4 bits).

In [9]:
# Exercice 2 : a completer -- code Shannon-Fano sur ABRACADABRA

corpus_ex2 = "ABRACADABRA"
freq_ex2 = Counter(corpus_ex2)
print(f"corpus : {corpus_ex2!r} ({len(corpus_ex2)} caracteres)")
print(f"frequences : {dict(freq_ex2)}")

# TODO etudiant : construire le code Shannon-Fano et afficher :
#   1. le code {symbole: 'bits'}
#   2. la longueur moyenne
# Utiliser shannon_fano(freq_ex2) ci-dessus.

# Indice : la fonction shannon_fano(freq) est deja disponible depuis la
# section 4. Inutile de la reimplementer -- seulement l'appeler et formater.


corpus : 'ABRACADABRA' (11 caracteres)
frequences : {'A': 5, 'B': 2, 'R': 2, 'C': 1, 'D': 1}


### Exercice 3 — Comparer Shannon-Fano et Huffman sur un corpus à 6 symboles

Construire un corpus `corpus_6` avec 6 symboles (A, B, C, D, E, F) de fréquences
respectives (0.30, 0.25, 0.15, 0.12, 0.10, 0.08) (la somme fait 1.0). Construire le
code Shannon-Fano et le code Huffman sur ce corpus, puis comparer leurs longueurs
moyennes et l'entropie $H$.

Afficher un tableau récapitulatif : `symbole | fréquence | code SF | code Huffman | $\ell_{SF}$ | $\ell_{Huf}$`.

**Indice** : Huffman utilisera un *tas binaire* (heapq) où chaque nœud est `(fréquence, id, [symboles])` —
reprendre la structure de la cellule de la section 5. L'écart attendu : Huffman ≤ Shannon-Fano.

In [10]:
# Exercice 3 : a completer -- SF vs Huffman sur 6 symboles

corpus_6 = {"A": 0.30, "B": 0.25, "C": 0.15, "D": 0.12, "E": 0.10, "F": 0.08}
# Verifier que la somme fait 1.0
assert abs(sum(corpus_6.values()) - 1.0) < 1e-9, "frequences doivent sommer a 1.0"
print(f"corpus_6 = {corpus_6}")

# TODO etudiant :
#   1. construire code_sf = shannon_fano(corpus_6)
#   2. construire code_huf = huffman_arbre(corpus_6)
#   3. calculer la longueur moyenne de chacun
#   4. calculer l'entropie H
#   5. afficher un tableau : symbole | freq | code SF | code Huffman | l_SF | l_Huf

# Indice : pour la longueur moyenne, reprendre le pattern de la cellule de
# la section 5 (sum sur freq[s] * len(code[s]) / total).


corpus_6 = {'A': 0.3, 'B': 0.25, 'C': 0.15, 'D': 0.12, 'E': 0.1, 'F': 0.08}


## Conclusion

Un message binaire n'est pas seulement une suite de bits : c'est un **langage** entre
l'encodeur et le décodeur. Trois conditions émergent de ce notebook :

1. **Le code doit être préfixe.** Sans cette condition, un message ne peut être décodé
   sans connaître sa longueur à l'avance. L'inégalité de Kraft donne une condition
   nécessaire sur les longueurs.
2. **Le code doit s'adapter aux fréquences.** Shannon-Fano et Huffman montrent qu'on
   peut raccourcir un message de moitié (voire plus) en attribuant des codes courts aux
   symboles fréquents. Huffman est strictement meilleur sur les corpus équilibrés.
3. **L'entropie borne la longueur moyenne.** Aucun code préfixe *caractère par caractère*
   ne peut descendre sous $H$. Cette borne est atteinte asymptotiquement par le codage
   par blocs (Shannon) ou par des techniques arithmétiques (au-delà de ce notebook).

Pour aller plus loin : voir **Compression-02** (Lempel-Ziv, famille différente qui
n'a pas besoin de connaître les fréquences à l'avance) et **Infer.NET-03** (codage
arithmétique pour les sources continues).
